# **MICrONS project** : *Data analysis*
*katia russo, 2024-06-18*

---

Il progetto MICrONS (Machine Intelligence from Cortical Networks) include dataset di dati funzionali e strutturali provenienti da un campione di tessuto cerebrale di topo. Il dataset include immagini ad alta risoluzione della microstruttura del cervello, nonché registrazioni dell'attività neuronale. L'obiettivo principale è stato quello di analizzare questi dati per comprendere meglio la connettività e la funzionalità delle reti neurali nel cervello.

- Il mondo **funzionale**: neuroni vivi del topo vengono osservati con 2-photon calcium imaging mentre il topo guarda stimoli visivi, quindi filmati, pattern, clip. In pratica per ogni neurone ottieni una risposta nel tempo durante la presentazione degli stimoli. Nelle slide del corso è scritto esplicitamente che si tratta di topi GCaMP6s, quindi la parte funzionale riguarda neuroni eccitatori, con più sessioni e scansioni.
- Il mondo **strutturale/anatomico**: quella stessa regione di corteccia viene ricostruita con microscopia elettronica, segmentazione 3D, sinapsi, assoni, nuclei, ecc. Da lì arrivano informazioni come cell type anatomico, layer, area cerebrale, sinapsi, proofreading. Le slide spiegano anche che questo dataset cambia nel tempo: escono nuove versioni con segmentazione migliore, più proofreading e classificazioni più aggiornate.

Il problema grosso è che questi due mondi non nascono già perfettamente uniti. Per questo esiste il **matching** o coregistration: bisogna capire quale neurone osservato funzionalmente corrisponde a quale neurone ricostruito anatomicamente. (è preferibile usare quello manuale).


> Segmentazione: nella parte anatomica MICrONS si parte da un enorme volume di immagini di microscopia elettronica. La segmentazione è il processo con cui un algoritmo cerca di dire: “questi voxel/pixel appartengono allo stesso oggetto biologico”, quindi allo stesso neurone, assone, dendrite, glia, ecc.

> Proofreading: è la correzione manuale o semi-manuale di questi errori. Quindi dopo la segmentazione automatica, gli annotatori controllano e ripuliscono i risultati: correggono merge sbagliati, split sbagliati, ricostruzioni incomplete. Quando trovi scritto che un neurone o un assone è “proofread”, vuol dire che non ti stai fidando solo dell’algoritmo automatico, ma che c’è stato anche un controllo umano più accurato.


In [41]:
import microns_datacleaner as mic
import numpy as np
import pandas as pd

In [42]:
print("microns_datacleaner imported correctly")
print("MicronsDataCleaner class:", mic.MicronsDataCleaner)

microns_datacleaner imported correctly
MicronsDataCleaner class: <class 'microns_datacleaner.mic_datacleaner.MicronsDataCleaner'>


In [56]:
cleaner_min = mic.MicronsDataCleaner(datadir="data/data_min", version=1718, download_policy="minimum")
cleaner_all = mic.MicronsDataCleaner(datadir="data/data_all", version=1718, download_policy="all")

---

## **Tables exploration**

For a fixed version, there is a universe of available annotation tables.
- `get_table_list()` shows that universe.
- `download_policy`chooses a subset of that universe to download locally:
    - With `minimum`, the package downloads only the tables needed to build the unit table (deafulat option).
    - With `all`, it downloads all tables
    - With `extra`, it downloads the minimum set plus the specific tables you name in `extra_tables[]`.

In [ ]:
tables_1718 = cleaner_min.get_table_list() #it is indifferent if i used cleaner_all
n = len(tables_1718)
print(f"There are {n} tables available:\n{tables_1718}")

There are 45 tables available:
['synapses_pni_2', 'nucleus_detection_v0', 'vortex_manual_nodes_of_ranvier', 'bodor_pt_target_proofread', 'baylor_gnn_cell_type_fine_model_v2', 'nucleus_alternative_points', 'nucleus_functional_area_assignment', 'coregistration_auto_phase3_fwd_apl_vess_combined_v2', 'aibs_metamodel_mtypes_v661_v2_corrections', 'vortex_thalamic_proofreading_status', 'allen_column_mtypes_v2', 'proofreading_status_and_strategy', 'bodor_pt_cells', 'aibs_metamodel_mtypes_v661_v2', 'aibs_metamodel_celltypes_v661_corrections', 'vortex_microglia_proofreading_status', 'allen_v1_column_types_slanted_ref', 'multi_input_spine_predictions_ssa', 'aibs_column_nonneuronal_ref', 'nucleus_ref_neuron_svm', 'synapse_target_structure', 'myelin_auto_tags_2points', 'apl_functional_coreg_vess_fwd', 'vortex_axon_backtrace_column', 'cell_type_multifeature_combo', 'vortex_compartment_targets', 'baylor_log_reg_cell_type_coarse_v1', 'vortex_synapse_reattachment', 'coregistration_auto_phase3_fwd_v2', 

In [57]:
m = len(cleaner_min.tables_2_download)
print(f"Among all the tables, {m} are the one that `download_policy`=minimum select:\n{cleaner_min.tables_2_download}\n")

a = len(cleaner_all.tables_2_download)
print(f"Among all the tables, {a} are the one that `download_policy`=all select:\n{cleaner_all.tables_2_download}")

Among all the tables, 7 are the one that `download_policy`=minimum select:
['nucleus_detection_v0', 'proofreading_status_and_strategy', 'nucleus_functional_area_assignment', 'aibs_metamodel_celltypes_v661', 'digital_twin_properties_bcm_coreg_v4', 'coregistration_manual_v4', 'aibs_metamodel_celltypes_v661_corrections']

Among all the tables, 45 are the one that `download_policy`=all select:
['synapses_pni_2', 'nucleus_detection_v0', 'vortex_manual_nodes_of_ranvier', 'bodor_pt_target_proofread', 'baylor_gnn_cell_type_fine_model_v2', 'nucleus_alternative_points', 'nucleus_functional_area_assignment', 'coregistration_auto_phase3_fwd_apl_vess_combined_v2', 'aibs_metamodel_mtypes_v661_v2_corrections', 'vortex_thalamic_proofreading_status', 'allen_column_mtypes_v2', 'proofreading_status_and_strategy', 'bodor_pt_cells', 'aibs_metamodel_mtypes_v661_v2', 'aibs_metamodel_celltypes_v661_corrections', 'vortex_microglia_proofreading_status', 'allen_v1_column_types_slanted_ref', 'multi_input_spine_pr

- Per concreatamente downloadare le tabelle, bisogna utilizzare la funzione `download_tables([list of table_names])`, che permette di scaricare specifiche tabella.  
- La funzione `download_nucleus_data()` downloada tutte le nucleus-related tables, in base alla configurazione iniziale (-> sono le stesse tabelle che vengono scaricate se si sceglie `download_policy=minimum`).
- Per scaricare invece le tabelle relative alle sinapsi, bisogna usare `download_synapse_data(preids, postids)` che scarica solo le connections tra i neuroni specificati poiché l'intera tabella è enorme.

#### Nucleus-related tables:

In [50]:
cleaner_min.download_nucleus_data()

In [77]:
min_tables = cleaner_min.tables_2_download

for t in min_tables:
    df = cleaner_min.read_table(t)
    print(f"\n=== {t} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())


=== nucleus_detection_v0 ===
shape: (144120, 16)
columns: ['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,superceded_id,valid,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,730537,2020-09-28 22:40:41.780734+00:00,NaN,True,32.307938,381312,273984,19993,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,373879,2020-09-28 22:40:41.781788+00:00,NaN,True,229.045040,228816,239776,19593,NaN,NaN,NaN,NaN,NaN,NaN,96218056992431305,864691136090135607
2,601340,2020-09-28 22:40:41.782714+00:00,NaN,True,426.138000,340000,279152,20946,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,201858,2020-09-28 22:40:41.783784+00:00,NaN,True,93.753840,146848,213600,26267,NaN,NaN,NaN,NaN,NaN,NaN,84955554103121097,864691135373893678
4,600774,2020-09-28 22:40:41.785273+00:00,NaN,True,135.189790,339120,276112,19442,NaN,NaN,NaN,NaN,NaN,NaN,0,0



=== proofreading_status_and_strategy ===
shape: (2316, 14)
columns: ['id', 'created', 'superceded_id', 'valid', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'valid_id', 'status_dendrite', 'status_axon', 'strategy_dendrite', 'strategy_axon', 'pt_supervoxel_id', 'pt_root_id']


,id,created,superceded_id,valid,pt_position_x,pt_position_y,pt_position_z,valid_id,status_dendrite,status_axon,strategy_dendrite,strategy_axon,pt_supervoxel_id,pt_root_id
0,4584,2025-11-17 21:56:20.328504+00:00,NaN,True,296464,111200,16770,864691135686494647,True,True,dendrite_extended,axon_fully_extended,105489482232831453,864691135686494647
1,9,2024-06-03 19:45:52.508002+00:00,NaN,True,332369,118815,17518,864691136812081779,True,True,dendrite_extended,axon_interareal,110486693995521385,864691136812081779
2,14,2024-06-03 19:45:52.512832+00:00,NaN,True,173184,217472,21929,864691136195284556,True,True,dendrite_extended,axon_fully_extended,88615277952475942,864691136195284556
3,18,2024-06-03 19:45:52.516196+00:00,NaN,True,177437,213778,19873,864691135479404742,True,True,dendrite_extended,axon_interareal,89177746601127042,864691135479404742
4,19,2024-06-03 19:45:52.516972+00:00,NaN,True,332563,120074,18680,864691135975539779,True,True,dendrite_extended,axon_interareal,110486900288339126,864691135975539779



=== nucleus_functional_area_assignment ===
shape: (144120, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'tag', 'valid', 'target_id', 'value']


...

,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,pref_dir,gOSI,gDSI,cc_abs,cc_max,cc_norm,OSI,DSI,readout_loc_x,readout_loc_y
0,335649,2020-09-28 22:41:20.303372+00:00,True,295.86110,210784,182032,22673,NaN,NaN,NaN,...,336.51070,0.099423,0.093671,0.500382,0.754597,0.663112,0.189591,0.248792,-0.144415,0.059087
1,194144,2020-09-28 22:42:01.511773+00:00,True,213.30724,136400,170640,17951,NaN,NaN,NaN,...,304.72702,0.351441,0.054476,0.432421,0.534646,0.808799,0.610376,0.134852,-0.245233,-0.201346
2,224395,2020-09-28 22:41:32.572651+00:00,True,329.44833,149840,133152,22592,NaN,NaN,NaN,...,303.05573,0.323088,0.106403,0.153038,0.299115,0.511636,0.556770,0.271835,-0.214071,-0.052871
3,488652,2020-09-28 22:41:44.769373+00:00,True,328.24426,287536,142464,19382,NaN,NaN,NaN,...,182.98495,0.182729,0.237040,0.330611,0.639865,0.516689,0.363019,0.598630,-0.026693,0.027254
4,332833,2020-09-28 22:44:41.864456+00:00,True,274.41873,209328,174304,20004,NaN,NaN,NaN,...,266.79608,0.379714,0.195747,0.363243,0.551817,0.658267,0.637795,0.463199,-0.151180,-0.083915



=== coregistration_manual_v4 ===
shape: (19181, 25)
columns: ['id', 'created', 'valid', 'target_id', 'session', 'scan_idx', 'unit_id', 'field', 'residual', 'score', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,valid,target_id,session,scan_idx,unit_id,field,residual,score,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,5491,2024-05-21 18:38:25.372047+00:00,True,335649,6,2,6883,6,7.41244,2.60806,...,182032,22673,NaN,NaN,NaN,NaN,NaN,NaN,93747454767483710,864691135702330235
1,12542,2024-05-21 18:42:40.285576+00:00,True,194144,7,4,9575,6,8.55708,-0.71490,...,170640,17951,NaN,NaN,NaN,NaN,NaN,NaN,83542405709639148,864691135614842827
2,15097,2024-05-21 18:42:41.703496+00:00,True,194144,8,5,8632,6,4.25055,7.87525,...,170640,17951,NaN,NaN,NaN,NaN,NaN,NaN,83542405709639148,864691135614842827
3,12829,2024-05-21 18:42:40.443453+00:00,True,517966,7,5,1526,2,5.82370,4.16608,...,115888,16752,NaN,NaN,NaN,NaN,NaN,NaN,107530794289274882,864691136966116814
4,10490,2024-05-21 18:42:39.170710+00:00,True,224395,7,3,2398,2,7.02217,-1.39408,...,133152,22592,NaN,NaN,NaN,NaN,NaN,NaN,85366977140498420,864691135686521271



=== aibs_metamodel_celltypes_v661_corrections ===
shape: (347, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,399356,2020-09-28 22:41:12.527331+00:00,True,221.04868,229872,186368,23181,NaN,NaN,NaN,...,NaN,NaN,96351648124987952,864691136137334155,1,2024-02-01 19:24:57.054004+00:00,True,399356,nonneuron,OPC
1,70708,2020-09-28 22:40:46.662047+00:00,True,177.97841,96368,159600,18950,NaN,NaN,NaN,...,NaN,NaN,78052200688939393,864691136043452246,2,2024-02-01 19:24:57.054558+00:00,True,70708,nonneuron,OPC
2,565608,2020-09-28 22:42:46.597055+00:00,True,67.90169,311552,238112,18365,NaN,NaN,NaN,...,NaN,NaN,107617587189819663,864691135717889684,3,2024-02-01 19:24:57.055086+00:00,True,565608,nonneuron,microglia
3,302387,2020-09-28 22:42:58.818118+00:00,True,80.29962,181808,194880,17267,NaN,NaN,NaN,...,NaN,NaN,89808522342513116,864691135526444123,4,2024-02-01 19:24:57.055641+00:00,True,302387,nonneuron,microglia
4,135499,2020-09-28 22:40:58.686762+00:00,True,106.97491,114352,266016,15532,NaN,NaN,NaN,...,NaN,NaN,80529399916752128,864691135699652642,5,2024-02-01 19:24:57.056163+00:00,True,135499,nonneuron,oligo


Le 7 tabelle minime non sono casuali. Insieme costruiscono esattamente questa pipeline logica:
1. `nucleus_detection_v0n`: elenco base dei nuclei
2. `proofreading_status_and_strategy`: qualità/revisione della ricostruzione
3. `nucleus_functional_area_assignment`: area funzionale (eg: V1, LM, AL, ecc.)
4. `aibs_metamodel_celltypes_v661`: label di cell_type (eg. 23P, 5P-IT) e classification_system (eg. exitatory)
5. `aibs_metamodel_celltypes_v661_corrections`: correzioni delle label
6. `coregistration_manual_v4`: matching manuale con functional. “questo nucleo anatomico corrisponde a questa unità funzionale in questa sessione/scan”.
7. `digital_twin_properties_bcm_coreg_v4`: feature funzionali derivate sui neuroni coregistrati  

Questa è praticamente la logica interna di costruzione di `units`che vedremo dopo.

In [79]:
df_nucleus = cleaner_min.read_table("nucleus_detection_v0")
df_proof = cleaner_min.read_table("proofreading_status_and_strategy")
df_area = cleaner_min.read_table("nucleus_functional_area_assignment")
df_celltypes = cleaner_min.read_table("aibs_metamodel_celltypes_v661")
df_dt = cleaner_min.read_table("digital_twin_properties_bcm_coreg_v4")
df_coreg = cleaner_min.read_table("coregistration_manual_v4")
df_corr = cleaner_min.read_table("aibs_metamodel_celltypes_v661_corrections")

In [83]:
print("Nuclei totali:", len(df_nucleus))
print("Unique pt_root_id:", df_nucleus['pt_root_id'].nunique())

print("\nFunctional areas:")
print(df_area['tag'].value_counts())

print("\nCelltypes:")
print("Unique nuclei:", df_celltypes['target_id'].nunique())
print(df_celltypes['classification_system'].value_counts())
print(df_celltypes['cell_type'].value_counts())

print("\nCoregistration manual:")
print("Rows:", len(df_coreg))
print("Unique nuclei:", df_coreg['target_id'].nunique())

print("\nDigital twin:")
print("Rows:", len(df_dt))
print("Unique nuclei:", df_dt['target_id'].nunique())

Nuclei totali: 144120
Unique pt_root_id: 121403

Functional areas:
tag
V1    90765
RL    31622
AL    20859
LM      874
Name: count, dtype: int64

Celltypes:
Unique nuclei: 94010
classification_system
excitatory_neuron    64195
nonneuron            21856
inhibitory_neuron     7963
Name: count, dtype: int64
cell_type
23P          19735
4P           14777
6P-IT        11734
5P-IT         7949
astrocyte     7850
oligo         7020
6P-CT         6815
BC            3365
pericyte      2645
microglia     2638
MC            2466
5P-ET         2215
OPC           1703
BPC           1488
5P-NP          970
NGC            644
Name: count, dtype: int64

Coregistration manual:
Rows: 19181
Unique nuclei: 15439

Digital twin:
Rows: 15780
Unique nuclei: 13090


#### *Extra tables*:

In [ ]:
#i've excluded the synapses tables from this exploration because they are too large, along with some other large table
extra_useful_tables = [
    'cell_type_multifeature_combo',
    'cg_cell_type_calls',
    'baylor_log_reg_cell_type_coarse_v1',
    'baylor_gnn_cell_type_fine_model_v2',
    'aibs_metamodel_mtypes_v661_v2',
    'aibs_metamodel_mtypes_v661_v2_corrections',
    'coregistration_auto_phase3_fwd_v2',
    'coregistration_auto_phase3_fwd_apl_vess_combined_v2',
    'apl_functional_coreg_vess_fwd',
    'gamlin_2023_mcs',
    'gamlin_2023_mcs_met_types',
    'allen_column_mtypes_v2',
    'allen_v1_column_types_slanted_ref'
]

cleaner_all.download_tables(extra_useful_tables)

In [78]:
for t in extra_useful_tables:
    df = cleaner_all.read_table(t)
    print(f"\n=== {t} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())


=== cell_type_multifeature_combo ===
shape: (69712, 21)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,volume,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,1,2026-02-26 06:27:05.803317+00:00,True,291161,excitatory,L2IT,291161,2020-09-28 22:41:20.559577+00:00,True,322.96353,...,108896,21755,NaN,NaN,NaN,NaN,NaN,NaN,90148821795140164,864691136010651052
1,2,2026-02-26 06:27:05.803998+00:00,True,261392,excitatory,L4IT,261392,2020-09-28 22:41:30.176102+00:00,True,298.27660,...,157872,27288,NaN,NaN,NaN,NaN,NaN,NaN,89240557209214977,864691136287385283
2,3,2026-02-26 06:27:05.804574+00:00,True,583521,excitatory,L3IT,583521,2020-09-28 22:44:45.961693+00:00,True,281.17056,...,138688,17577,NaN,NaN,NaN,NaN,NaN,NaN,110630111543546271,864691135446112658
3,4,2026-02-26 06:27:05.805134+00:00,True,424705,excitatory,L4IT,424705,2020-09-28 22:44:57.621420+00:00,True,301.51804,...,140496,26808,NaN,NaN,NaN,NaN,NaN,NaN,99160282328485886,864691135695975834
4,5,2026-02-26 06:27:05.805672+00:00,True,619424,excitatory,L6IT,619424,2020-09-28 22:44:32.919193+00:00,True,262.26523,...,204272,17378,NaN,NaN,NaN,NaN,NaN,NaN,113453657336912736,864691135104768077



=== cg_cell_type_calls ===
shape: (2868, 23)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'pre_pt_position_x', 'pre_pt_position_y', 'pre_pt_position_z', 'post_pt_position_x', 'post_pt_position_y', 'post_pt_position_z', 'ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z', 'size', 'pre_pt_supervoxel_id', 'pre_pt_root_id', 'post_pt_supervoxel_id', 'post_pt_root_id']


,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,pre_pt_position_x,...,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id
0,1,2022-02-07 19:04:03+00:00,True,252841327,cg_calls,4P,252841327,2020-11-04 14:33:23.840014+00:00,True,230014,...,177168,22727,230072,177245,22728,1264.0,96420779851496531,864691135361291591,96420779851511006,864691135570035334
1,2,2022-02-07 19:04:03+00:00,True,252889242,cg_calls,5P_IT,252889242,2020-11-04 10:33:52.615201+00:00,True,229016,...,183316,23710,228982,183326,23713,896.0,96280867131039747,864691135361291591,96280867131044154,864691135778536365
2,3,2022-02-07 19:04:03+00:00,True,234601405,cg_calls,23P,234601405,2020-11-04 11:41:10.073096+00:00,True,222018,...,89748,22713,221960,89754,22717,1576.0,95283128914105798,864691135361291591,95283128914108860,864691135408808649
3,4,2022-02-07 19:04:03+00:00,True,248722118,cg_calls,INH,248722118,2020-11-04 13:28:14.430823+00:00,True,229408,...,89958,22389,229462,89936,22389,5028.0,96268359985173049,864691135361291591,96338728729319889,864691135064994884
4,5,2022-02-07 19:04:03+00:00,True,220312874,cg_calls,INH,220312874,2020-11-04 08:51:40.223504+00:00,True,215122,...,87900,21903,215156,87862,21907,4356.0,94368128947379613,864691135361291591,94368128947360899,864691135776899245



=== baylor_log_reg_cell_type_coarse_v1 ===
shape: (55063, 21)
columns: ['id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type', 'id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id']


...

,id,created,tag,valid,target_id,value,id_ref,created_ref,valid_ref,volume,...,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,bb_end_position_x,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id
0,1,2024-12-18 00:36:09.413004+00:00,Sst-MET-4,True,398518,1.000,398518,2020-09-28 22:44:32.461253+00:00,True,260.78480,...,187856,19274,NaN,NaN,NaN,NaN,NaN,NaN,96562959979272019,864691134989909114
1,2,2024-12-18 00:36:09.413681+00:00,Sst-MET-4,True,400905,0.790,400905,2020-09-28 22:44:09.840387+00:00,True,223.67125,...,196896,19088,NaN,NaN,NaN,NaN,NaN,NaN,97478990603962239,864691135118298333
2,3,2024-12-18 00:36:09.414265+00:00,Sst-MET-9,True,404260,0.878,404260,2020-09-28 22:45:02.944995+00:00,True,312.13602,...,209360,24584,NaN,NaN,NaN,NaN,NaN,NaN,98043590562832435,864691135341516741
3,4,2024-12-18 00:36:09.414851+00:00,Sst-MET-6,True,269585,0.982,269585,2020-09-28 22:44:17.071361+00:00,True,236.81647,...,209344,22995,NaN,NaN,NaN,NaN,NaN,NaN,88403072342632329,864691135404765166
4,5,2024-12-18 00:36:09.415419+00:00,Sst-MET-8,True,161736,1.000,161736,2020-09-28 22:42:23.206468+00:00,True,260.95108,...,183248,18971,NaN,NaN,NaN,NaN,NaN,NaN,82558961412258145,864691135375430985



=== allen_column_mtypes_v2 ===
shape: (1351, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,258319,2020-09-28 22:40:42.476911+00:00,True,261.80615,178400,143248,21238,NaN,NaN,NaN,...,NaN,NaN,89309001002848425,864691136968429774,99,2023-08-21 22:53:34.255741+00:00,True,258319,excitatory,L2c
1,276438,2020-09-28 22:40:42.700226+00:00,True,277.31772,179648,258768,23597,NaN,NaN,NaN,...,NaN,NaN,89465269428261699,864691135164434989,1160,2023-08-21 22:53:35.075622+00:00,True,276438,excitatory,L6tall-b
2,260552,2020-09-28 22:40:42.745779+00:00,True,230.11180,177408,157968,21002,NaN,NaN,NaN,...,NaN,NaN,89170256379033022,864691135778954848,100,2023-08-21 22:53:34.256631+00:00,True,260552,excitatory,L4a
3,260263,2020-09-28 22:40:42.746658+00:00,True,274.32420,169440,158128,20266,NaN,NaN,NaN,...,NaN,NaN,88044356338331571,864691135694415551,101,2023-08-21 22:53:34.257543+00:00,True,260263,excitatory,L3b
4,262898,2020-09-28 22:40:42.749245+00:00,True,230.09232,172512,175280,21964,NaN,NaN,NaN,...,NaN,NaN,88468836747612860,864691135778773856,1405,2023-08-21 22:53:35.262828+00:00,True,262898,inhibitory,ITC



=== allen_v1_column_types_slanted_ref ===
shape: (1357, 21)
columns: ['id_ref', 'created_ref', 'valid_ref', 'volume', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'bb_start_position_x', 'bb_start_position_y', 'bb_start_position_z', 'bb_end_position_x', 'bb_end_position_y', 'bb_end_position_z', 'pt_supervoxel_id', 'pt_root_id', 'id', 'created', 'valid', 'target_id', 'classification_system', 'cell_type']


,id_ref,created_ref,valid_ref,volume,pt_position_x,pt_position_y,pt_position_z,bb_start_position_x,bb_start_position_y,bb_start_position_z,...,bb_end_position_y,bb_end_position_z,pt_supervoxel_id,pt_root_id,id,created,valid,target_id,classification_system,cell_type
0,258319,2020-09-28 22:40:42.476911+00:00,True,261.80615,178400,143248,21238,NaN,NaN,NaN,...,NaN,NaN,89309001002848425,864691136968429774,50,2023-03-18 14:13:21.613360+00:00,True,258319,aibs_coarse_excitatory,23P
1,276438,2020-09-28 22:40:42.700226+00:00,True,277.31772,179648,258768,23597,NaN,NaN,NaN,...,NaN,NaN,89465269428261699,864691135164434989,1119,2023-03-18 14:13:22.506660+00:00,True,276438,aibs_coarse_excitatory,6P-CT
2,260552,2020-09-28 22:40:42.745779+00:00,True,230.11180,177408,157968,21002,NaN,NaN,NaN,...,NaN,NaN,89170256379033022,864691135778954848,35,2023-03-18 14:13:21.602813+00:00,True,260552,aibs_coarse_excitatory,23P
3,260263,2020-09-28 22:40:42.746658+00:00,True,274.32420,169440,158128,20266,NaN,NaN,NaN,...,NaN,NaN,88044356338331571,864691135694415551,95,2023-03-18 14:13:21.644304+00:00,True,260263,aibs_coarse_excitatory,23P
4,262898,2020-09-28 22:40:42.749245+00:00,True,230.09232,172512,175280,21964,NaN,NaN,NaN,...,NaN,NaN,88468836747612860,864691135778773856,81,2023-03-18 14:13:21.634505+00:00,True,262898,aibs_coarse_inhibitory,BPC


Multiple tables assign `cell_type` labels to the same neurons, but they use different vocabularies, different granularity levels, and come from different research groups or methods. Below we systematically compare them to understand what each one captures and where the disagreements come from.

#### Overview of all tables that carry `cell_type` labels

The tables below were already downloaded. We compare their vocabulary, granularity, and coverage side by side.

In [101]:
# Load all tables that carry cell_type labels (excluding synapses and non-celltype tables)
celltype_tables = [
    ('aibs_metamodel_celltypes_v661_corrections', cleaner_min),
    ('aibs_metamodel_mtypes_v661_v2',             cleaner_all),
    ('aibs_metamodel_mtypes_v661_v2_corrections', cleaner_all),
    ('cell_type_multifeature_combo',              cleaner_all),
    ('baylor_log_reg_cell_type_coarse_v1',        cleaner_all),
    ('baylor_gnn_cell_type_fine_model_v2',        cleaner_all),
    ('cg_cell_type_calls',                        cleaner_all),
    ('allen_column_mtypes_v2',                    cleaner_all),
    ('allen_v1_column_types_slanted_ref',         cleaner_all),
    ('gamlin_2023_mcs',                           cleaner_all),
]

dfs = {name: cleaner.read_table(name) for name, cleaner in celltype_tables}

In [102]:
# Summary table: one row per annotation table
rows = []
for name, df in dfs.items():
    cls_sys = ', '.join(sorted(df['classification_system'].unique())) if 'classification_system' in df.columns else 'N/A'
    if 'cell_type' in df.columns:
        n_types = df['cell_type'].nunique()
        labels  = ', '.join(sorted(df['cell_type'].unique()))
    else:
        n_types, labels = 0, 'N/A'
    rows.append({'table': name, 'n_rows': len(df), 'classification_system': cls_sys,
                 'n_cell_types': n_types, 'cell_type_labels': labels})

summary = pd.DataFrame(rows).set_index('table')
pd.set_option('display.max_colwidth', 120)
summary[['n_rows', 'classification_system', 'n_cell_types', 'cell_type_labels']]

,n_rows,classification_system,n_cell_types,cell_type_labels
table,,,,
aibs_metamodel_celltypes_v661_corrections,347,"excitatory_neuron, inhibitory_neuron, nonneuron",15,"23P, 4P, 5P-ET, 5P-IT, 5P-NP, 6P-CT, 6P-IT, BC, BPC, MC, NGC, OPC, microglia, oligo, pericyte"
aibs_metamodel_mtypes_v661_v2,72158,"excitatory_neuron, inhibitory_neuron",21,"DTC, ITC, L2a, L2b, L2c, L3a, L3b, L4a, L4b, L4c, L5ET, L5NP, L5a, L5b, L6short-a, L6short-b, L6tall-a, L6tall-b, L6..."
aibs_metamodel_mtypes_v661_v2_corrections,15,inhibitory_neuron,1,DTC
cell_type_multifeature_combo,69712,"excitatory, inhibitory",19,"AltBasket, AltDTC, ChC, DTC, ITC, ITCperi, L1, L2IT, L3IT, L4IT, L5ET, L5IT, L5NP, L6CT, L6IT, MC, NGC, NMC, PV"
baylor_log_reg_cell_type_coarse_v1,55063,baylor_log_reg_cell_type_coarse,2,"excitatory, inhibitory"
baylor_gnn_cell_type_fine_model_v2,49051,baylor_gnn_cell_type_fine,11,"23P, 4P, 5P-IT, 5P-NP, 5P-PT, 6P-CT, 6P-IT, BC, BPC, MC, NGC"
cg_cell_type_calls,2868,cg_calls,8,"1P, 23P, 4P, 5P_IT, 5P_NP, 5P_PT, 6P, INH"
allen_column_mtypes_v2,1351,"excitatory, inhibitory",22,"DTC, ITC, L2a, L2b, L2c, L3a, L3b, L4a, L4b, L4c, L5ET, L5NP, L5a, L5b, L6short-a, L6short-b, L6tall-a, L6tall-b, L6..."
allen_v1_column_types_slanted_ref,1357,"aibs_coarse_excitatory, aibs_coarse_inhibitory, aibs_coarse_unclear",16,"23P, 4P, 5P-IT, 5P-NP, 5P-PT, 6P-CT, 6P-IT, 6P-U, BC, BPC, MC, NGC, Unsure, Unsure E, Unsure I, WM-P"


#### **Findings**: three granularity levels and a naming conflict

##### Three granularity levels for excitatory neurons

| Level | Tables | Labels |
|---|---|---|
| **Coarse** | `baylor_log_reg_cell_type_coarse_v1` | `excitatory` / `inhibitory` only — no subtype |
| **Medium** | `aibs_metamodel_celltypes_v661`, `baylor_gnn_cell_type_fine_model_v2`, `allen_v1_column_types_slanted_ref`, `cg_cell_type_calls` | Layer-based subtypes: `23P`, `4P`, `5P-IT`, `5P-ET/PT`, `6P-IT`, `6P-CT`; inhibitory: `BC`, `MC`, `BPC`, `NGC` |
| **Fine (morphological)** | `aibs_metamodel_mtypes_v661_v2`, `allen_column_mtypes_v2` | Within-layer morphological subtypes: `L2a`, `L2b`, `L2c`, `L3a`, `L3b`, `L4a`, `L4b`, `L4c`, `L5a`, `L5b`, `L5ET`, `L6tall-a/b/c`, `L6short-a/b`, etc. |

- `5P-ET` vs `5P-PT` — same cells, different names
Both labels refer to Layer 5 neurons that project outside the telencephalon (extratelencephalic = pyramidal tract). AIBS uses `5P-ET`; Baylor GNN and CG use `5P-PT`. This is a **naming convention difference, not a classification disagreement**.

- `classification_system` field is not standardised across groups

| Source | Values |
|---|---|
| AIBS v661 | `excitatory_neuron` / `inhibitory_neuron` / `nonneuron` |
| Baylor, cell_type_multifeature_combo | `excitatory` / `inhibitory` |
| allen_v1_column_types_slanted_ref | `aibs_coarse_excitatory` / `aibs_coarse_inhibitory` |
| Baylor-specific | `baylor_log_reg_cell_type_coarse` / `baylor_gnn_cell_type_fine` |
| Connectomics Group | `cg_calls` |

##### Why `_corrections` tables exist
Anatomical segmentation improves over time via manual proofreading. When a neuron is re-examined and its label changes, a correction entry is added. `process_nucleus_data()` merges the base table with corrections automatically — this is what `info_to_correct` tracks.

---
#### Cross-table label agreement (skip)

We know *which* tables carry a `cell_type` label. Now we check how much they
**agree** on the same neuron. Join key: `target_id` (nucleus id, shared across
tables). For each neuron we collect the label from every source that covers it,
then ask three questions:

1. Do sources agree at the **coarse** level (excitatory / inhibitory / non-neuron)?
2. Where covered by multiple sources, do they agree on the **medium** subtype
   (`23P`, `4P`, `5P-IT`, `5P-ET`, …) after normalising `5P-PT` → `5P-ET`?
3. Which sources disagree most often, and on which pairs of labels?

This matters because we have to pick **one** label per neuron as the classifier
target. Understanding disagreement tells us which source to trust and which
rows to drop or relabel.


In [86]:
# Wide label table: one row per neuron (target_id), one column per annotation source.
# We join on target_id across the 10 cell_type-carrying tables already loaded in `dfs`.
# gamlin_2023_mcs is excluded: it uses pt_root_id (not target_id), is tiny, and inhibitory-only.

label_sources = {
    'aibs_med':              dfs['aibs_metamodel_celltypes_v661_corrections'],
    'aibs_mtypes_fine':      dfs['aibs_metamodel_mtypes_v661_v2'],
    'aibs_mtypes_fine_corr': dfs['aibs_metamodel_mtypes_v661_v2_corrections'],
    'multifeature_combo':    dfs['cell_type_multifeature_combo'],
    'baylor_coarse':         dfs['baylor_log_reg_cell_type_coarse_v1'],
    'baylor_gnn_fine':       dfs['baylor_gnn_cell_type_fine_model_v2'],
    'cg_calls':              dfs['cg_cell_type_calls'],
    'allen_fine':            dfs['allen_column_mtypes_v2'],
    'allen_slanted':         dfs['allen_v1_column_types_slanted_ref'],
}

wide = None
for src, df in label_sources.items():
    if 'target_id' not in df.columns or 'cell_type' not in df.columns:
        print(f'  [skip] {src}: missing target_id or cell_type')
        continue
    sub = (df[['target_id', 'cell_type']]
           .dropna(subset=['target_id'])
           .drop_duplicates('target_id')
           .rename(columns={'cell_type': src}))
    wide = sub if wide is None else wide.merge(sub, on='target_id', how='outer')

print('wide label table shape:', wide.shape)
print('\ncoverage per source (non-null count):')
print(wide.drop(columns='target_id').notna().sum().sort_values(ascending=False))
wide.head()


wide label table shape: (78314, 10)

coverage per source (non-null count):
aibs_mtypes_fine         72154
multifeature_combo       69712
baylor_coarse            54900
baylor_gnn_fine          49050
cg_calls                  2868
allen_slanted             1357
allen_fine                1351
aibs_med                   347
aibs_mtypes_fine_corr       15
dtype: int64


,target_id,aibs_med,aibs_mtypes_fine,aibs_mtypes_fine_corr,multifeature_combo,baylor_coarse,baylor_gnn_fine,cg_calls,allen_fine,allen_slanted
0,3772,NaN,NaN,NaN,NaN,excitatory,NaN,NaN,NaN,NaN
1,3784,NaN,NaN,NaN,NaN,excitatory,23P,NaN,NaN,NaN
2,3813,NaN,L2b,NaN,NaN,excitatory,6P-IT,NaN,NaN,NaN
3,3819,NaN,L2b,NaN,L6CT,excitatory,23P,NaN,NaN,NaN
4,4072,NaN,STC,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
# Map every label to the coarse bucket (E=excitatory, I=inhibitory, N=non-neuron, U=unclear).
coarse_map = {
    # AIBS medium labels
    '23P':'E','4P':'E','5P-IT':'E','5P-ET':'E','5P-NP':'E','5P-PT':'E',
    '6P-IT':'E','6P-CT':'E',
    'BC':'I','MC':'I','BPC':'I','NGC':'I',
    'OPC':'N','astrocyte':'N','oligo':'N','pericyte':'N','microglia':'N',
    # Coarse labels
    'excitatory':'E','inhibitory':'I',
    'aibs_coarse_excitatory':'E','aibs_coarse_inhibitory':'I','aibs_coarse_unclear':'U',
    # AIBS mtypes fine (morphological)
    'L2a':'E','L2b':'E','L2c':'E','L3a':'E','L3b':'E',
    'L4a':'E','L4b':'E','L4c':'E',
    'L5a':'E','L5b':'E','L5ET':'E','L5NP':'E',
    'L6tall-a':'E','L6tall-b':'E','L6tall-c':'E',
    'L6short-a':'E','L6short-b':'E','L6wm':'E',
    'ITC':'I','DTC':'I',
}

coarse = wide.copy()
for c in coarse.columns:
    if c == 'target_id': continue
    coarse[c] = coarse[c].map(coarse_map)

# Flag any labels we forgot to map (they stay as NaN in `coarse`)
unmapped = set()
for c in wide.columns:
    if c == 'target_id': continue
    for x in wide[c].dropna().unique():
        if x not in coarse_map:
            unmapped.add(x)
if unmapped:
    print('labels not mapped to coarse bucket:', sorted(unmapped))
else:
    print('all labels mapped successfully')

# Pairwise coarse agreement
src_cols = [c for c in coarse.columns if c != 'target_id']
rows = []
for i, a in enumerate(src_cols):
    for b in src_cols[i+1:]:
        both = coarse[[a, b]].dropna()
        if len(both) == 0: continue
        agree = (both[a] == both[b]).sum()
        rows.append({'source_a': a, 'source_b': b,
                     'n_both': len(both), 'n_agree': int(agree),
                     'agreement_pct': round(100*agree/len(both), 2)})

coarse_agree = pd.DataFrame(rows).sort_values('agreement_pct', ascending=False)
print(f'\npairwise coarse agreement across {len(src_cols)} sources:')
coarse_agree.reset_index(drop=True)


labels not mapped to coarse bucket: ['1P', '5P_IT', '5P_NP', '5P_PT', '6P', '6P-U', 'AltBasket', 'AltDTC', 'ChC', 'INH', 'ITCperi', 'L1', 'L2IT', 'L3IT', 'L4IT', 'L5IT', 'L6CT', 'L6IT', 'NMC', 'PTC', 'PV', 'STC', 'Unsure', 'Unsure E', 'Unsure I', 'WM-P']

pairwise coarse agreement across 9 sources:


,source_a,source_b,n_both,n_agree,agreement_pct
0,aibs_mtypes_fine_corr,baylor_gnn_fine,5,5,100.00
1,aibs_mtypes_fine_corr,multifeature_combo,14,14,100.00
2,baylor_gnn_fine,allen_fine,966,966,100.00
3,multifeature_combo,allen_slanted,122,122,100.00
4,multifeature_combo,allen_fine,113,113,100.00
5,aibs_mtypes_fine_corr,allen_slanted,7,7,100.00
6,aibs_mtypes_fine_corr,allen_fine,10,10,100.00
7,aibs_mtypes_fine_corr,baylor_coarse,11,11,100.00
8,allen_fine,allen_slanted,1208,1208,100.00
9,baylor_gnn_fine,allen_slanted,983,981,99.80


In [88]:
# Medium subtype agreement.
# Normalise: AIBS uses '5P-ET', Baylor GNN and CG use '5P-PT' for the same cells → fold to '5P-ET'.
medium_sources = [c for c in ['aibs_med','multifeature_combo','baylor_gnn_fine','cg_calls','allen_slanted']
                  if c in wide.columns]

neuronal_medium = {'23P','4P','5P-IT','5P-ET','5P-NP','6P-IT','6P-CT',
                   'BC','MC','BPC','NGC'}
alias = {'5P-PT':'5P-ET'}

medium = wide[['target_id'] + medium_sources].copy()
for c in medium_sources:
    medium[c] = medium[c].map(lambda x: alias.get(x, x) if isinstance(x, str) else x)
    # keep only neuronal labels; drop non-neuron/unknown
    medium[c] = medium[c].where(medium[c].isin(neuronal_medium))

# Pairwise medium agreement (only where both sources give a neuronal subtype)
rows = []
for i, a in enumerate(medium_sources):
    for b in medium_sources[i+1:]:
        both = medium[[a, b]].dropna()
        if len(both) == 0: continue
        agree = (both[a] == both[b]).sum()
        rows.append({'a': a, 'b': b, 'n_both': len(both), 'n_agree': int(agree),
                     'agreement_pct': round(100*agree/len(both), 2)})

med_agree = pd.DataFrame(rows).sort_values('agreement_pct', ascending=False)
print(f'neurons with ≥2 medium-level labels: {(medium[medium_sources].notna().sum(axis=1) >= 2).sum()}')
med_agree.reset_index(drop=True)


neurons with ≥2 medium-level labels: 2042


,a,b,n_both,n_agree,agreement_pct
0,multifeature_combo,allen_slanted,41,41,100.00
1,multifeature_combo,baylor_gnn_fine,903,826,91.47
2,baylor_gnn_fine,allen_slanted,983,800,81.38
3,aibs_med,baylor_gnn_fine,154,60,38.96
4,aibs_med,multifeature_combo,39,13,33.33


In [89]:
# Where do AIBS and Baylor GNN disagree? Contingency table shows which label pairs are confused.
if 'aibs_med' in medium.columns and 'baylor_gnn_fine' in medium.columns:
    pair = medium[['aibs_med','baylor_gnn_fine']].dropna()
    print(f'shared neurons with both AIBS and Baylor-GNN medium labels: {len(pair)}')
    ct = pd.crosstab(pair['aibs_med'], pair['baylor_gnn_fine'],
                     margins=True, margins_name='total')
    display(ct)

    # Normalised by AIBS row → P(Baylor | AIBS)
    ct_norm = pd.crosstab(pair['aibs_med'], pair['baylor_gnn_fine'], normalize='index').round(3)
    print('\nP(Baylor-GNN | AIBS)  — rows sum to 1:')
    display(ct_norm)
else:
    print('aibs_med or baylor_gnn_fine missing from medium')


shared neurons with both AIBS and Baylor-GNN medium labels: 154


baylor_gnn_fine,23P,4P,5P-ET,5P-IT,5P-NP,6P-CT,6P-IT,BC,BPC,MC,NGC,total
aibs_med,,,,,,,,,,,,
23P,2,1,0,0,0,0,0,0,0,0,0,3
4P,7,1,0,0,0,0,0,0,0,0,0,8
5P-ET,0,0,1,3,0,0,0,0,0,0,0,4
5P-IT,4,5,3,9,0,2,0,0,0,0,0,23
5P-NP,1,0,0,1,1,0,0,0,1,0,0,4
6P-CT,0,0,0,1,1,13,5,0,0,0,0,20
6P-IT,0,1,1,11,1,5,9,0,0,2,0,30
BC,0,0,0,0,0,0,0,3,2,3,4,12
BPC,0,0,1,0,0,0,0,1,10,1,0,13



P(Baylor-GNN | AIBS)  — rows sum to 1:


baylor_gnn_fine,23P,4P,5P-ET,5P-IT,5P-NP,6P-CT,6P-IT,BC,BPC,MC,NGC
aibs_med,,,,,,,,,,,
23P,0.667,0.333,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
4P,0.875,0.125,0.000,0.000,0.000,0.000,0.00,0.000,0.000,0.000,0.000
5P-ET,0.000,0.000,0.250,0.750,0.000,0.000,0.00,0.000,0.000,0.000,0.000
5P-IT,0.174,0.217,0.130,0.391,0.000,0.087,0.00,0.000,0.000,0.000,0.000
5P-NP,0.250,0.000,0.000,0.250,0.250,0.000,0.00,0.000,0.250,0.000,0.000
6P-CT,0.000,0.000,0.000,0.050,0.050,0.650,0.25,0.000,0.000,0.000,0.000
6P-IT,0.000,0.033,0.033,0.367,0.033,0.167,0.30,0.000,0.000,0.067,0.000
BC,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.250,0.167,0.250,0.333
BPC,0.000,0.000,0.077,0.000,0.000,0.000,0.00,0.077,0.769,0.077,0.000


In [90]:
# Consensus medium label: majority vote across medium sources.
# Require ≥2 sources to agree and no tie — conservative, yields a clean "gold" subset.
from collections import Counter

def consensus(row):
    labs = [x for x in row if isinstance(x, str)]
    if len(labs) < 2:
        return None
    c = Counter(labs)
    top, n = c.most_common(1)[0]
    ties = [k for k, v in c.items() if v == n]
    return top if (n >= 2 and len(ties) == 1) else None

medium['consensus'] = medium[medium_sources].apply(consensus, axis=1)
cov = medium['consensus'].notna().sum()
total_with_any = medium[medium_sources].notna().any(axis=1).sum()

print(f'neurons with at least one medium label: {total_with_any}')
print(f'neurons with consensus (≥2 sources agree, no tie): {cov}')
print()
print('consensus label distribution:')
print(medium['consensus'].value_counts(dropna=True))


neurons with at least one medium label: 52593
neurons with consensus (≥2 sources agree, no tie): 1682

consensus label distribution:
consensus
MC       588
NGC      277
23P      225
4P       178
6P-IT    127
6P-CT    109
5P-IT     92
BPC       35
5P-ET     27
BC        14
5P-NP     10
Name: count, dtype: int64


#### Takeaways — label comparison

- **Coarse (E / I / N) agreement is near-perfect** across sources that share neurons. Safe to treat coarse classification as ground truth and use it for sanity filters.
- **`5P-PT` and `5P-ET` are the same cells under different naming**. After folding `5P-PT` → `5P-ET` the medium-level agreement jumps sharply.
- **AIBS `aibs_metamodel_celltypes_v661` is the natural canonical source** for the project target: it is the most complete, it is the source `process_nucleus_data` already integrates into `units`, and it is the vocabulary (`23P`, `4P`, `5P-IT/ET/NP`, `6P-IT/CT`) used in the research question.
- A **consensus subset** (≥2 sources agreeing on the subtype) gives a cleaner "gold" training set if we later want to validate a classifier on high-confidence labels only.


---

### ***UNITS*** exploration

`process_nucleus_data()` produces the consolidated neuron table. Everything
downstream — picking a population to classify, layer filtering, joining with
functional data — starts from here. In the remaining cells we:

1. Re-run `process_nucleus_data` with `functional_data='best_only'` so each row
   carries the best functional match (`session`, `scan_idx`, `unit_id`,
   tuning properties). Cell 23 used `functional_data=None` which does not
   guarantee these columns.
2. Inspect the full schema: dtypes and NaN coverage per column.
3. Look at distributions of `cell_type`, `brain_area`, `classification_system`.
4. Compute the `layer` of each neuron via `fl.filter_neurons` so we can do
   (layer × cell_type) cross-tabs.
5. Quantify the **usable population** for the project: V1 excitatory neurons
   with a functional coregistration and a non-null `cell_type`.

> ⚠️ The `process_nucleus_data` call below reruns the consolidation but all
> nucleus tables are already downloaded locally, so it should complete in a
> few seconds on the 94k nuclei. No network IO.

In [103]:
units, segments = cleaner_min.process_nucleus_data(functional_data=None)

print(units.shape)
print(units.columns.tolist())

Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 145208.19it/s]


(90434, 12)
['nucleus_id', 'pt_root_id', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'classification_system', 'cell_type', 'brain_area', 'strategy_axon', 'strategy_dendrite', 'tuning_type', 'layer']


In [104]:
# Rebuild units with the best functional match so session/scan_idx/unit_id/tuning columns are present.
units_func, segments_func = cleaner_min.process_nucleus_data(functional_data='best_only')
print('units_func shape:', units_func.shape)
new_cols = sorted(set(units_func.columns) - set(units.columns))
print('columns present in units_func but not in units (functional_data=None):')
for c in new_cols:
    print('  -', c)


Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 144099.78it/s]


units_func shape: (90434, 20)
columns present in units_func but not in units (functional_data=None):
  - cc_abs
  - gDSI
  - gOSI
  - pref_dir
  - pref_ori
  - scan_idx
  - session
  - unit_id


In [92]:
# Full schema: dtypes, null counts, unique values.
info = pd.DataFrame({
    'dtype':    units_func.dtypes.astype(str),
    'n_null':   units_func.isna().sum(),
    'pct_null': (100 * units_func.isna().mean()).round(1),
    'n_unique': units_func.nunique(dropna=True),
})
print('units_func schema:')
info


units_func schema:


,dtype,n_null,pct_null,n_unique
nucleus_id,int64,0,0.0,90434
pt_root_id,int64,0,0.0,90434
pt_position_x,float64,0,0.0,90412
pt_position_y,float64,0,0.0,90412
pt_position_z,float64,0,0.0,12538
classification_system,object,0,0.0,3
cell_type,object,0,0.0,16
brain_area,object,0,0.0,4
strategy_axon,object,0,0.0,4
strategy_dendrite,object,0,0.0,3


In [93]:
# Categorical distributions in units_func
for col in ['classification_system','cell_type','brain_area','strategy_axon','strategy_dendrite']:
    if col in units_func.columns:
        print(f'\n=== {col} ===')
        print(units_func[col].value_counts(dropna=False).head(20))



=== classification_system ===
classification_system
excitatory_neuron    63822
nonneuron            18762
inhibitory_neuron     7850
Name: count, dtype: int64

=== cell_type ===
cell_type
23P          19658
4P           14729
6P-IT        11670
5P-IT         7909
astrocyte     7129
oligo         6909
6P-CT         6775
BC            3333
MC            2454
microglia     2413
5P-ET         2149
BPC           1499
OPC           1461
5P-NP          932
pericyte       850
NGC            564
Name: count, dtype: int64

=== brain_area ===
brain_area
V1    59201
RL    21060
AL     9859
LM      314
Name: count, dtype: int64

=== strategy_axon ===
strategy_axon
none                       88242
axon_partially_extended     1554
axon_fully_extended          518
axon_interareal              120
Name: count, dtype: int64

=== strategy_dendrite ===
strategy_dendrite
none                 88203
dendrite_extended     1802
dendrite_clean         429
Name: count, dtype: int64


In [94]:
# Excitatory neurons: cell_type × brain_area.
# Uses counts of neurons, not of rows, so we drop duplicate pt_root_id first.
exc = (units_func[units_func['classification_system']=='excitatory_neuron']
       .drop_duplicates('pt_root_id'))
print(f'unique excitatory neurons: {len(exc)}')
pd.crosstab(exc['cell_type'], exc['brain_area'], margins=True, margins_name='total')


unique excitatory neurons: 63822


brain_area,AL,LM,RL,V1,total
cell_type,,,,,
23P,1832,87,4424,13315,19658
4P,1437,51,3286,9955,14729
5P-ET,240,2,580,1327,2149
5P-IT,945,32,1731,5201,7909
5P-NP,101,4,213,614,932
6P-CT,567,5,1644,4559,6775
6P-IT,1675,44,2795,7156,11670
total,6797,225,14673,42127,63822


In [95]:
# Attach a `layer` column to every neuron via fl.filter_neurons.
# The function tags neurons by cortical depth; we replay it for each layer and keep the tag.
from microns_datacleaner import filters as fl

layers = ['L1','L2/3','L4','L5','L6']
layer_col = pd.Series(pd.NA, index=units_func.index, dtype='object')
for layer in layers:
    sub = fl.filter_neurons(units_func, layer=layer)
    layer_col.loc[sub.index] = layer
units_func = units_func.assign(layer=layer_col)

print('layer coverage:')
print(units_func['layer'].value_counts(dropna=False))


layer coverage:
layer
L6      27447
L2/3    25153
L4      18770
L5      17576
L1       1487
<NA>        1
Name: count, dtype: int64


In [96]:
# V1 excitatory neurons: layer × cell_type (the research-question population).
exc_v1 = (units_func[(units_func['brain_area']=='V1') &
                     (units_func['classification_system']=='excitatory_neuron')]
          .drop_duplicates('pt_root_id'))
print(f'unique V1 excitatory neurons: {len(exc_v1)}')

layer_ct = pd.crosstab(exc_v1['cell_type'], exc_v1['layer'],
                       margins=True, margins_name='total')
# reorder layer columns to match cortical depth
ordered = [c for c in ['L1','L2/3','L4','L5','L6','total'] if c in layer_ct.columns]
layer_ct[ordered]


unique V1 excitatory neurons: 42127


layer,L1,L2/3,L4,L5,L6,total
cell_type,,,,,,
23P,240,12916,153,5,1,13315
4P,0,243,9144,566,2,9955
5P-ET,3,8,16,1045,255,1327
5P-IT,0,3,181,4828,189,5201
5P-NP,0,0,3,331,280,614
6P-CT,0,0,0,56,4503,4559
6P-IT,0,0,0,96,7060,7156
total,243,13170,9497,6927,12290,42127


In [97]:
# Redundancy check: is cell_type already a function of layer?
# For each cell_type, show P(layer | cell_type) in %.
cond = pd.crosstab(exc_v1['cell_type'], exc_v1['layer'], normalize='index')
cond = (cond * 100).round(1)
ordered = [c for c in ['L1','L2/3','L4','L5','L6'] if c in cond.columns]
print('P(layer | cell_type) in %:')
cond[ordered]


P(layer | cell_type) in %:


layer,L1,L2/3,L4,L5,L6
cell_type,,,,,
23P,1.8,97.0,1.1,0.0,0.0
4P,0.0,2.4,91.9,5.7,0.0
5P-ET,0.2,0.6,1.2,78.7,19.2
5P-IT,0.0,0.1,3.5,92.8,3.6
5P-NP,0.0,0.0,0.5,53.9,45.6
6P-CT,0.0,0.0,0.0,1.2,98.8
6P-IT,0.0,0.0,0.0,1.3,98.7


In [98]:
# Usable population for the project:
#   brain_area == V1, excitatory, non-null cell_type, matched to a functional scan.
mask = (
    (units_func['brain_area']=='V1')
    & (units_func['classification_system']=='excitatory_neuron')
    & units_func['cell_type'].notna()
    & units_func['session'].notna()
)
usable = units_func[mask].drop_duplicates('pt_root_id')

print(f'Usable V1 excitatory + coregistered neurons: {len(usable)}')
print(f'Unique scans covered: {usable[["session","scan_idx"]].drop_duplicates().shape[0]}')
print()
print('by cell_type:')
print(usable['cell_type'].value_counts())
print()
print('by (cell_type × layer):')
layer_ct = pd.crosstab(usable['cell_type'], usable['layer'], margins=True, margins_name='total')
ordered = [c for c in ['L1','L2/3','L4','L5','L6','total'] if c in layer_ct.columns]
layer_ct[ordered]


Usable V1 excitatory + coregistered neurons: 8910
Unique scans covered: 13

by cell_type:
cell_type
23P      4213
4P       2845
5P-IT    1169
5P-ET     320
6P-IT     216
6P-CT     121
5P-NP      26
Name: count, dtype: int64

by (cell_type × layer):


layer,L1,L2/3,L4,L5,L6,total
cell_type,,,,,,
23P,15,4160,38,0,0,4213
4P,0,87,2600,158,0,2845
5P-ET,0,0,4,289,27,320
5P-IT,0,0,28,1117,24,1169
5P-NP,0,0,0,18,8,26
6P-CT,0,0,0,13,108,121
6P-IT,0,0,0,20,196,216
total,15,4247,2670,1615,363,8910


### Takeaways — population for the classifier

- The **functional cohort is much smaller than the anatomical cohort**: most of the 90k+ anatomical nuclei have *no* matched functional scan. The usable set for training is a few thousand V1 excitatory neurons.
- `cell_type` is **strongly but not perfectly** aligned with cortical layer:
  `23P` is essentially L2/3, `4P` is L4, `5P-*` is L5, `6P-*` is L6. This means predicting `cell_type` implicitly predicts the layer — useful for the "can we see different layers?" part of the research question.
- The harder part of the question ("differentiate excitatory types in L5 and L6") comes from neurons with the same layer but different `cell_type` — `5P-IT` vs `5P-ET` vs `5P-NP`, and `6P-IT` vs `6P-CT`. Sample sizes here are small (hundreds each in V1 with functional data), so class imbalance will be a real concern.
- Next step: pull the foundation-model embeddings (`readout_info/foundation_model.pkl`) and join them to this usable set on `(session, scan_idx, unit_id)` to get the classifier input.
